# Error Handling

These exercises practice the error handling of the course: catching an error with `try`/`except`, one `except` clause per error, `else` and `finally`, raising an error with `raise`, defining an application error and wrapping another one in it. The first three exercises are independent; the last three build on each other, up to reading a measurement file some of whose lines are invalid. When an exercise asks you to predict a result, write your prediction down as a comment before running the cell, then compare.

## Converting Without Crashing

A form returns the temperatures its users typed, as strings, in the list `entries` below. Write a function `to_number(text)` that returns `text` converted to a `float`, or, if the conversion fails, prints “Value skipped:” followed by the representation of the error, and returns `None`. Catch only the error the conversion raises. Then print the list of the results of `to_number` on each entry.

Before running your cell, predict as a comment which entries fail.

In [ ]:
entries = ["21.5", "-3", "twenty", "", "1e3", "12,5"]

In [ ]:
# Your code here

### Additional hints

`float("twenty")` raises a `ValueError`: that's the class to name in the `except`, and `as e` gives access to the error, which `{e!r}` prints with its class. A list comprehension applies `to_number` to each entry. You should get `[21.5, -3.0, None, None, 1000.0, None]`: `float` reads scientific notation (`1e3`), not a decimal comma.

```python
def to_number(text: str) -> float | None:
  try:
    return ...
  except ... as e:
    print(f"Value skipped: {...}")
    return None


print([to_number(text) for text in entries])  # Prediction: ...
```

### Solution

In [ ]:
def to_number(text: str) -> float | None:
  try:
    return float(text)
  except ValueError as e:
    print(f"Value skipped: {e!r}")
    return None


# "twenty", "" and "12,5" fail; "1e3" is 1000.0
print([to_number(text) for text in entries])

## One `except` Clause per Error

Write a function `change(before, after)` that prints the change, as a percentage, between two values given as strings: $(\text{after} - \text{before}) / \text{before} \times 100$, with its sign and one decimal (“Change: +15.0 %”). Two errors each have their own message:

- a value that isn't a number: “Not a number:” followed by the error's message;
- a starting value of zero: “Undefined change: starting value is zero”.

The `try` block holds only what can fail: printing the result goes in an `else`. Call `change` on `"80"` and `"92"`, on `"80"` and `"n/a"`, then on `"0"` and `"5"`.

Before running your cell, predict as a comment the message that `change("0", "abc")` prints, then check.

In [ ]:
# Your code here

### Additional hints

Converting the two strings can raise a `ValueError`, dividing by zero a `ZeroDivisionError`: two `except` clauses, each with its message. `print(e)`, or `{e}` in an f-string, prints the message of the error `e`. The `else` only runs if the `try` went all the way through. In an f-string, the `+.1f` format specification prints the sign and one decimal. The statements of the `try` run in order: the first one that fails picks the `except` clause.

```python
def change(before: str, after: str) -> None:
  try:
    start = float(before)
    end = ...
    rate = ...
  except ValueError as e:
    print(f"Not a number: {e}")
  except ...:
    print(...)
  else:
    print(f"Change: {rate:...} %")


change("80", "92")
change("80", "n/a")
change("0", "5")
change("0", "abc")  # Prediction: ...
```

### Solution

In [ ]:
def change(before: str, after: str) -> None:
  try:
    start = float(before)
    end = float(after)
    rate = (end - start) / start * 100
  except ValueError as e:
    print(f"Not a number: {e}")
  except ZeroDivisionError:
    print("Undefined change: starting value is zero")
  else:
    print(f"Change: {rate:+.1f} %")


change("80", "92")
change("80", "n/a")
change("0", "5")
# float("abc") fails before the division: "Not a number"
change("0", "abc")

## `finally`: Whatever Happens

Write a function `count_lines(path)` that returns the number of lines of the text file `path`. If the file doesn't exist, it prints “File not found:” followed by the path, and returns `0`. In every case, it prints “Done reading” followed by the path, with a `finally` block. Print the result of `count_lines` on `log.txt`, which the next cell creates, then on `missing.txt`, which doesn't exist.

Before running your cell, predict as a comment the order of its five printed lines: those of the two calls and those of the two `print`s of the result.

In [ ]:
%%writefile log.txt
2026-10-12 08:00 startup
2026-10-12 08:05 kitchen OK
2026-10-12 08:07 cellar OK

In [ ]:
# Your code here

### Additional hints

`open` raises a `FileNotFoundError` when the file doesn't exist. `file.readlines()` returns the list of the lines, whose number `len` gives. The `finally` runs even when the `try` or the `except` leaves the function with a `return`: just before the function hands back its value. You should find 3 lines in `log.txt`.

```python
def count_lines(path: str) -> int:
  try:
    with open(path, encoding="utf-8") as file:
      return ...
  except ...:
    print(...)
    return 0
  finally:
    print(...)


print(count_lines("log.txt"))
print(count_lines("missing.txt"))
# Prediction: ...
```

### Solution

In [ ]:
def count_lines(path: str) -> int:
  try:
    with open(path, encoding="utf-8") as file:
      return len(file.readlines())
  except FileNotFoundError:
    print(f"File not found: {path}")
    return 0
  finally:
    print(f"Done reading {path}")


# The finally prints its message before print shows the result:
# Done reading log.txt, 3, File not found: missing.txt,
# Done reading missing.txt, 0
print(count_lines("log.txt"))
print(count_lines("missing.txt"))

## Raising an Error

The last three exercises read a measurement file each line of which reads `sensor;value`, for example `kitchen;21.5`. Write a function `read_measure(line)` that returns the tuple `(sensor, value)` of a line, the value as a `float`. It raises a `ValueError`, with a message that says what's wrong:

- if the line doesn't have exactly two fields: “expected 2 fields, found 3”;
- if the value isn't between -40 and 60 °C: “value out of range: 85.0”.

A value that isn't a number already raises a `ValueError`: let it propagate. Then try `read_measure` on each line of `test_lines`: print the measure, or the line and the error's message.

**Useful documentation**

- [`str.split`](https://docs.python.org/3/library/stdtypes.html#str.split)

In [ ]:
test_lines = ["kitchen;21.5", "cellar;12;3", "attic;85", "garden;abc"]

In [ ]:
# Your code here

### Additional hints

`line.split(";")` cuts the line at each `;` and returns the list of the fields. `raise ValueError(f"...")` raises the error with its message, and leaves the function. For the trial, a `try` around the call, printing the measure in the `else`. You should get one measure (`kitchen`) and three rejected lines, each for a different reason.

```python
def read_measure(line: str) -> tuple[str, float]:
  fields = line.split(";")
  if ...:
    raise ValueError(f"expected 2 fields, found {...}")
  sensor, text = fields
  value = float(text)
  if ...:
    raise ...
  return sensor, value


for line in test_lines:
  try:
    sensor, value = ...
  except ValueError as e:
    print(f"{line!r} rejected: {e}")
  else:
    print(f"{sensor}: {value} °C")
```

### Solution

In [ ]:
def read_measure(line: str) -> tuple[str, float]:
  fields = line.split(";")
  if len(fields) != 2:
    raise ValueError(f"expected 2 fields, found {len(fields)}")
  sensor, text = fields
  value = float(text)  # raises ValueError if text isn't a number
  if value < -40 or value > 60:
    raise ValueError(f"value out of range: {value}")
  return sensor, value


for line in test_lines:
  try:
    sensor, value = read_measure(line)
  except ValueError as e:
    print(f"{line!r} rejected: {e}")
  else:
    print(f"{sensor}: {value} °C")

## An Application Error

The rest of the program shouldn't have to know the errors `read_measure` raises. Define a custom error `InvalidMeasureError`, which inherits from `Exception`, with a docstring. Then write `read_checked_measure(line)`, which returns the result of `read_measure(line)`, but turns any `ValueError` into an `InvalidMeasureError`, with the message “invalid line 'cellar;12;3' (expected 2 fields, found 3)”, keeping the `ValueError` as its cause.

1. Catch the `InvalidMeasureError` of `read_checked_measure("cellar;12;3")` and print its message.
2. Predict as a comment whether an `except ValueError` catches the error of `read_checked_measure("attic;85")`, then check it with two `except` clauses, `ValueError` then `InvalidMeasureError`, each printing which one caught the error.
3. Call `read_checked_measure("cellar;12;3")` outside any `try`, and read the traceback: two errors, linked by “The above exception was the direct cause of the following exception”. Then comment that call out, so that the cell runs without an error.

In [ ]:
# Your code here

### Additional hints

A custom error is an empty class that inherits from `Exception`: its docstring is enough of a body. In `read_checked_measure`, a `try` around the call, and in the `except ValueError as e`, `raise ... from e` raises the new error with `e` as its cause. `InvalidMeasureError` inherits from `Exception`, not from `ValueError`: look at the course's hierarchy.

```python
class InvalidMeasureError(...):
  """..."""


def read_checked_measure(line: str) -> tuple[str, float]:
  try:
    return ...
  except ValueError as e:
    raise InvalidMeasureError(f"invalid line {line!r} ({e})") from ...


try:
  read_checked_measure("cellar;12;3")
except ... as e:
  print(f"Measure refused: {e}")

# Prediction: ...
try:
  read_checked_measure("attic;85")
except ValueError:
  print("Caught by except ValueError")
except ...:
  print(...)

# read_checked_measure("cellar;12;3")
```

### Solution

In [ ]:
class InvalidMeasureError(Exception):
  """A line of the measurement file the application can't read."""


def read_checked_measure(line: str) -> tuple[str, float]:
  try:
    return read_measure(line)
  except ValueError as e:
    raise InvalidMeasureError(f"invalid line {line!r} ({e})") from e


# 1. The application's error, with its cause's message
try:
  read_checked_measure("cellar;12;3")
except InvalidMeasureError as e:
  print(f"Measure refused: {e}")

# 2. InvalidMeasureError inherits from Exception, not from ValueError:
# except ValueError doesn't catch it
try:
  read_checked_measure("attic;85")
except ValueError:
  print("Caught by except ValueError")
except InvalidMeasureError:
  print("Caught by except InvalidMeasureError")

# 3. The traceback shows the ValueError, then the InvalidMeasureError it caused
# read_checked_measure("cellar;12;3")

## Reading a Measurement File

The next cell creates the file `measures.txt`, three lines of which are invalid. Write a function `read_file(path)` that reads the file line by line with `read_checked_measure` and returns a dictionary that maps each sensor to the list of its values. An invalid line doesn't stop the reading: the function prints “Line 4 skipped:” followed by the error's message, and moves on to the next one. Then print, for each sensor, its number of measures and their average with one decimal.

Finally, call `read_file` on `measures-2025.txt`, which doesn't exist, and print “File not found:” followed by the error's message, without the cell stopping on an error. Where should that error be caught, in `read_file` or around its call? Answer as a comment.

In [ ]:
%%writefile measures.txt
kitchen;21.5
cellar;12.0
kitchen;22.5
attic;85
cellar;11,5
garden;-2.5
kitchen
garden;-3.5
cellar;13.0

In [ ]:
# Your code here

### Additional hints

`enumerate(file, start=1)` gives each line with its number, starting from 1. `line.strip()` removes the trailing line break before the reading. The `try` surrounds only the call to `read_checked_measure`, adding to the dictionary goes in the `else`. An error is caught where you know how to handle it: an invalid line in the loop, to move on to the next one; a missing file, in the caller, which knows what to do without data. You should get lines 4, 5 and 7 skipped, then `kitchen` (2 measures, 22.0 °C), `cellar` (2 measures, 12.5 °C) and `garden` (2 measures, -3.0 °C).

```python
def read_file(path: str) -> dict[str, list[float]]:
  measures = {}
  with open(path, encoding="utf-8") as file:
    for number, line in enumerate(file, start=1):
      try:
        sensor, value = ...
      except ... as e:
        print(...)
      else:
        if sensor not in measures:
          measures[sensor] = []
        ...
  return measures


measures = read_file("measures.txt")
for sensor, values in measures.items():
  average = ...
  print(f"{sensor}: {len(values)} measures, average {average:.1f} °C")

# Answer: ...
try:
  read_file("measures-2025.txt")
except ... as e:
  print(...)
```

### Solution

In [ ]:
def read_file(path: str) -> dict[str, list[float]]:
  measures = {}
  with open(path, encoding="utf-8") as file:
    for number, line in enumerate(file, start=1):
      try:
        sensor, value = read_checked_measure(line.strip())
      except InvalidMeasureError as e:
        print(f"Line {number} skipped: {e}")
      else:
        if sensor not in measures:
          measures[sensor] = []
        measures[sensor].append(value)
  return measures


measures = read_file("measures.txt")
for sensor, values in measures.items():
  average = sum(values) / len(values)
  print(f"{sensor}: {len(values)} measures, average {average:.1f} °C")

# Around the call: read_file doesn't know what to do without a file,
# the caller does (warn the user, take another file…)
try:
  read_file("measures-2025.txt")
except FileNotFoundError as e:
  print(f"File not found: {e}")